# Cours 08 : annoter un texte avec Flair

Ce notebook reprend la séance en Python : on annote *Faux visage*, puis on explore les annotations avec des tableaux et des graphiques.

Il se lance **depuis le dossier `cours_08`**, avec l'environnement `env` activé et le modèle rangé dans `model/` (voir [01_installation.md](01_installation.md)) :

```bash
pip install notebook
jupyter notebook cours_08.ipynb
```

Exécutez les cellules dans l'ordre (`Maj + Entrée`).

## 1. Charger le modèle

On réutilise les fonctions du script `annoter.py` : `charger` (le modèle) et `decouper` (le texte en phrases et en mots). Le chargement prend de quelques secondes à une minute.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import torch, flair
from annoter import charger, decouper, NIVEAUX

flair.device = torch.device("cpu")
taches = charger("model/model-v6.pt")
[t.tag_type for t in taches]

## 2. Annoter une phrase

Chaque tâche ajoute ses étiquettes à la phrase : `pos` sur chaque mot, les autres sur des groupes de mots (les entités).

In [ ]:
def annoter(phrases):
    for tache in taches:
        tache.predict(phrases, mini_batch_size=8, embedding_storage_mode="cpu")
    return phrases

phrase = annoter(decouper("Le Roy partit de Paris le 3 iuin 1650 auec Monſieur le Cardinal Mazarin."))[0]
pd.DataFrame([(mot.text, mot.get_label("pos").value) for mot in phrase], columns=["mot", "pos"]).T

In [ ]:
pd.DataFrame([(niveau, span.text, span.get_label(niveau).value, round(span.get_label(niveau).score, 2))
              for niveau in NIVEAUX for span in phrase.get_spans(niveau)],
             columns=["niveau", "entité", "type", "score"])

## 3. Annoter tout le texte

Comme en ligne de commande, on recolle d'abord les mots coupés en fin de ligne (`¬`). L'annotation prend **2 à 5 minutes**.

In [ ]:
texte = open("data/faux_visage.txt", encoding="utf-8").read().replace("¬ ", "")
phrases = []
for ligne in texte.splitlines():
    phrases.extend(decouper(ligne))
print(len(phrases), "phrases")
annoter(phrases)
print("terminé")

On range les résultats dans deux tableaux pandas, l'équivalent des deux fichiers TSV de la séance.

In [ ]:
pos = pd.DataFrame([(n, mot.text, mot.get_label("pos").value)
                    for n, p in enumerate(phrases, start=1) for mot in p],
                   columns=["phrase", "mot", "pos"])
entites = pd.DataFrame([(n, niveau, span.text, span.get_label(niveau).value, span.get_label(niveau).score)
                        for n, p in enumerate(phrases, start=1)
                        for niveau in NIVEAUX for span in p.get_spans(niveau)],
                       columns=["phrase", "niveau", "entite", "type", "score"])
print(len(pos), "mots,", len(entites), "annotations d'entités")
entites.head()

## 4. Les parties du discours

In [ ]:
pos["pos"].value_counts().head(15).plot.barh()
plt.gca().invert_yaxis()
plt.title("Les 15 étiquettes les plus fréquentes")
plt.show()

Les noms communs les plus fréquents (l'équivalent de `awk` + `cut` + `sort` + `uniq -c`) :

In [ ]:
pos[pos["pos"] == "Nc"]["mot"].value_counts().head(15)

## 5. Les entités nommées

In [ ]:
coarse = entites[entites["niveau"] == "coarse"]
coarse["type"].value_counts()

In [ ]:
coarse[coarse["type"] == "pers"]["entite"].value_counts().head(20)

Les variantes graphiques sont comptées à part (*Henry*, *Henry de Valloys*, *DE VALOYS*...).
On peut les regrouper à la main avec un dictionnaire : c'est le début d'un **index des personnes**.

In [ ]:
regroupement = {
    "Henry": "Henri III", "Henry de Valloys": "Henri III", "DE VALOYS": "Henri III",
    "VALOYS": "Henri III", "ENRY DE VALLOYS": "Henri III", "Henry de Nauarre": "Henri de Navarre",
    "Ieſus-Chriſt": "Jésus-Christ", "Ieſus Chriſt": "Jésus-Christ", "Ieſus": "Jésus-Christ",
}
personnes = coarse[coarse["type"] == "pers"]["entite"].replace(regroupement)
personnes.value_counts().head(10)

## 6. Voir les entités dans le texte

On colore les entités du niveau `coarse` directement dans une phrase.

In [ ]:
from IPython.display import HTML

COULEURS = {"pers": "#ffd6a5", "loc": "#caffbf", "time": "#9bf6ff", "func": "#bdb2ff",
            "org": "#ffc6ff", "event": "#fdffb6", "prod": "#e0e0e0", "amount": "#e0e0e0"}

def colorer(phrase):
    # Recompose la phrase mot à mot, en colorant les entités
    entites = {span.tokens[0].idx: span for span in phrase.get_spans("coarse")}
    html, i = "", 1
    while i <= len(phrase):
        if i in entites:
            span = entites[i]
            etiquette = span.get_label("coarse").value
            html += (f'<mark style="background:{COULEURS.get(etiquette, "#eee")}">{span.text} '
                     f'<small><b>{etiquette}</b></small></mark>')
            dernier = span.tokens[-1]
        else:
            dernier = phrase[i - 1]
            html += dernier.text
        html += " " if dernier.whitespace_after else ""
        i = dernier.idx + 1
    return html

HTML("<br><br>".join(colorer(p) for p in phrases[30:36]))

## Exercices

1. Dans la section 2, annotez une phrase de votre choix (un texte du XVIIe siècle, un article de presse...). Le modèle s'en sort-il sur du français moderne ?
2. Dans la section 4, affichez les 15 verbes conjugués (`Vvc`) les plus fréquents.
3. Complétez le dictionnaire de la section 5 pour regrouper *France* et *Frãce*, *Paris* et *P Aris*. Attention : *Frãce* est classé parmi les personnes. Comment le corriger dans le tableau `coarse` ?
4. Dans la section 6, changez `phrases[30:36]` pour parcourir d'autres passages, et relevez une erreur du modèle.